# 🥇 Capa Gold: Jobs Data Engineering - Star Schema

## 🎯 Objetivo
Crear un modelo dimensional tipo **Star Schema** para análisis de ofertas de trabajo, con tablas de dimensiones y hechos.

## 🏗️ Arquitectura Star Schema

```
           ┌─────────────────┐
           │  dim_employer   │
           └────────┬────────┘
                    │
           ┌────────▼────────┐
           │   fact_jobs     │◄─────────┐
           └────────┬────────┘          │
                    │                   │
        ┌───────────┴───────────┐      │
        │                       │      │
┌───────▼────────┐    ┌────────▼──────▼──┐
│ dim_location   │    │ dim_employment_type │
└────────────────┘    └─────────────────────┘
```

## 📋 Tablas a Crear

### 🔷 Dimensiones (dim_*)
1. **dim_employer** - Información de empleadores
2. **dim_location** - Dimensión geográfica (país, estado, ciudad)
3. **dim_employment_type** - Tipos de empleo (Full-time, Contractor, etc.)
4. **dim_date** - Dimensión de fechas (año, mes, día, trimestre, semestre, día de la semana)

### 🔶 Hechos (fact_*)
1. **fact_jobs** - Tabla de hechos principal con métricas de cada oferta
2. **fact_jobs_summary** - Agregaciones y KPIs del mercado

---

**Fuente:** `prueba_api.silver.jobs_silver`  
**Destino:** `prueba_api.gold.dim_*` y `prueba_api.gold.fact_*`  
**Método:** Persistencia directa desde Silver (sin vistas temporales)

In [0]:

-- Insertar/actualizar datos en dim_employer de forma idempotente usando MERGE
-- Se puede ejecutar múltiples veces sin duplicar registros

MERGE INTO prueba_api.gold.dim_employer AS target
USING (
    SELECT 
        employer_name,
        employer_logo,
        employer_website,
        employer_reviews,
        CURRENT_TIMESTAMP() AS fecha_carga
    FROM (
        SELECT DISTINCT
            employer_name,
            FIRST_VALUE(employer_logo) OVER (PARTITION BY employer_name ORDER BY processed_at DESC) AS employer_logo,
            FIRST_VALUE(employer_website) OVER (PARTITION BY employer_name ORDER BY processed_at DESC) AS employer_website,
            FIRST_VALUE(employer_reviews) OVER (PARTITION BY employer_name ORDER BY processed_at DESC) AS employer_reviews
        FROM prueba_api.silver.jobs_silver
    ) unique_employers
) AS source
ON target.employer_name = source.employer_name

-- Si el empleador ya existe, actualizar sus datos
WHEN MATCHED THEN
    UPDATE SET
        target.employer_logo = source.employer_logo,
        target.employer_website = source.employer_website,
        target.employer_reviews = source.employer_reviews,
        target.fecha_carga = source.fecha_carga

-- Si el empleador no existe, insertarlo
WHEN NOT MATCHED THEN
    INSERT (employer_name, employer_logo, employer_website, employer_reviews, fecha_carga)
    VALUES (source.employer_name, source.employer_logo, source.employer_website, source.employer_reviews, source.fecha_carga);

In [0]:
-- Verificar cuántos empleadores tienen sitio web después del enriquecimiento
SELECT 
    COUNT(*) AS total_empleadores,
    COUNT(employer_website) AS empleadores_con_web,
    COUNT(*) - COUNT(employer_website) AS empleadores_sin_web,
    ROUND(COUNT(employer_website) * 100.0 / COUNT(*), 2) AS pct_con_web
FROM prueba_api.gold.dim_employer;

In [0]:
-- Mostrar los empleadores principales que ahora tienen sitio web
SELECT 
    e.employer_name,
    e.employer_website,
    COUNT(f.job_id) AS total_ofertas
FROM prueba_api.gold.dim_employer e
LEFT JOIN prueba_api.gold.fact_jobs f ON e.employer_sk = f.employer_sk
WHERE e.employer_website IS NOT NULL
GROUP BY e.employer_name, e.employer_website
ORDER BY total_ofertas DESC, e.employer_name
LIMIT 20;

In [0]:

-- Insertar/actualizar datos en dim_location de forma idempotente usando MERGE
-- Se puede ejecutar múltiples veces sin duplicar registros

MERGE INTO prueba_api.gold.dim_location AS target
USING (
    SELECT 
        job_country,
        job_state,
        job_city,
        job_location AS location_full_text,
        ubicacion_completa,
        job_latitude,
        job_longitude,
        CURRENT_TIMESTAMP() AS fecha_carga
    FROM (
        SELECT DISTINCT
            job_country,
            job_state,
            job_city,
            FIRST_VALUE(job_location) OVER (PARTITION BY job_country, job_state, job_city ORDER BY processed_at DESC) AS job_location,
            FIRST_VALUE(ubicacion_completa) OVER (PARTITION BY job_country, job_state, job_city ORDER BY processed_at DESC) AS ubicacion_completa,
            FIRST_VALUE(job_latitude) OVER (PARTITION BY job_country, job_state, job_city ORDER BY processed_at DESC) AS job_latitude,
            FIRST_VALUE(job_longitude) OVER (PARTITION BY job_country, job_state, job_city ORDER BY processed_at DESC) AS job_longitude
        FROM prueba_api.silver.jobs_silver
    ) unique_locations
) AS source
ON COALESCE(target.job_country, '') = COALESCE(source.job_country, '')
   AND COALESCE(target.job_state, '') = COALESCE(source.job_state, '')
   AND COALESCE(target.job_city, '') = COALESCE(source.job_city, '')

-- Si la ubicación ya existe, actualizar sus datos
WHEN MATCHED THEN
    UPDATE SET
        target.location_full_text = source.location_full_text,
        target.ubicacion_completa = source.ubicacion_completa,
        target.job_latitude = source.job_latitude,
        target.job_longitude = source.job_longitude,
        target.fecha_carga = source.fecha_carga

-- Si la ubicación no existe, insertarla
WHEN NOT MATCHED THEN
    INSERT (job_country, job_state, job_city, location_full_text, ubicacion_completa, job_latitude, job_longitude, fecha_carga)
    VALUES (source.job_country, source.job_state, source.job_city, source.location_full_text, source.ubicacion_completa, source.job_latitude, source.job_longitude, source.fecha_carga);

In [0]:

-- Insertar/actualizar datos en dim_employment_type de forma idempotente usando MERGE
-- Se puede ejecutar múltiples veces sin duplicar registros

MERGE INTO prueba_api.gold.dim_employment_type AS target
USING (
    SELECT 
        job_employment_type,
        CASE 
            WHEN job_employment_type = 'Full-time' THEN 'Tiempo Completo'
            WHEN job_employment_type = 'Contractor' THEN 'Contratista'
            WHEN job_employment_type = 'Part-time' THEN 'Medio Tiempo'
            ELSE job_employment_type
        END AS employment_type_es,
        CURRENT_TIMESTAMP() AS fecha_carga
    FROM (
        SELECT DISTINCT job_employment_type
        FROM prueba_api.silver.jobs_silver
        WHERE job_employment_type IS NOT NULL
    ) unique_types
) AS source
ON target.job_employment_type = source.job_employment_type

-- Si el tipo de empleo ya existe, actualizar sus datos
WHEN MATCHED THEN
    UPDATE SET
        target.employment_type_es = source.employment_type_es,
        target.fecha_carga = source.fecha_carga

-- Si el tipo de empleo no existe, insertarlo
WHEN NOT MATCHED THEN
    INSERT (job_employment_type, employment_type_es, fecha_carga)
    VALUES (source.job_employment_type, source.employment_type_es, source.fecha_carga);

In [0]:

-- Insertar/actualizar niveles de seniority desde Silver de forma idempotente usando MERGE
MERGE INTO prueba_api.gold.dim_seniority AS target
USING (
    SELECT DISTINCT
        seniority_level,
        CASE seniority_level
            WHEN 'No especificado' THEN 'No especificado'
            WHEN 'Entry Level' THEN 'Nivel de Entrada'
            WHEN 'Junior' THEN 'Junior'
            WHEN 'Mid-level' THEN 'Nivel Medio'
            WHEN 'Senior' THEN 'Senior'
            WHEN 'Lead/Staff' THEN 'Líder/Staff'
            WHEN 'Executive' THEN 'Ejecutivo'
            ELSE seniority_level
        END AS seniority_level_es,
        CASE seniority_level
            WHEN 'No especificado' THEN 0
            WHEN 'Entry Level' THEN 1
            WHEN 'Junior' THEN 2
            WHEN 'Mid-level' THEN 3
            WHEN 'Senior' THEN 4
            WHEN 'Lead/Staff' THEN 5
            WHEN 'Executive' THEN 6
            ELSE 0
        END AS seniority_order,
        CURRENT_TIMESTAMP() AS fecha_carga
    FROM prueba_api.silver.jobs_silver
    WHERE seniority_level IS NOT NULL
) AS source
ON target.seniority_level = source.seniority_level

-- Si el nivel de seniority ya existe, actualizar sus datos
WHEN MATCHED THEN
    UPDATE SET
        target.seniority_level_es = source.seniority_level_es,
        target.seniority_order = source.seniority_order,
        target.fecha_carga = source.fecha_carga

-- Si el nivel de seniority no existe, insertarlo
WHEN NOT MATCHED THEN
    INSERT (seniority_level, seniority_level_es, seniority_order, fecha_carga)
    VALUES (source.seniority_level, source.seniority_level_es, source.seniority_order, source.fecha_carga);

In [0]:
-- Crear dimensión de fechas con clave surrogada autoincremental
CREATE OR REPLACE TABLE prueba_api.gold.dim_date (
    date_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    fecha DATE,
    anio INT,
    mes INT,
    dia INT,
    nombre_mes STRING,
    nombre_mes_corto STRING,
    trimestre INT,
    semestre INT,
    dia_semana INT,
    nombre_dia_semana STRING,
    nombre_dia_semana_corto STRING,
    semana_anio INT,
    dia_anio INT,
    es_fin_semana BOOLEAN,
    es_primer_dia_mes BOOLEAN,
    es_ultimo_dia_mes BOOLEAN,
    fecha_carga TIMESTAMP
)
COMMENT 'Dimensión de fechas con clave surrogada autoincremental (IDENTITY)';

-- Insertar/actualizar fechas desde Silver de forma idempotente usando MERGE
MERGE INTO prueba_api.gold.dim_date AS target
USING (
    SELECT DISTINCT
        CAST(job_posted_at_datetime_utc AS DATE) AS fecha,
        YEAR(job_posted_at_datetime_utc) AS anio,
        MONTH(job_posted_at_datetime_utc) AS mes,
        DAY(job_posted_at_datetime_utc) AS dia,
        DATE_FORMAT(job_posted_at_datetime_utc, 'MMMM') AS nombre_mes,
        DATE_FORMAT(job_posted_at_datetime_utc, 'MMM') AS nombre_mes_corto,
        QUARTER(job_posted_at_datetime_utc) AS trimestre,
        CASE 
            WHEN MONTH(job_posted_at_datetime_utc) <= 6 THEN 1 
            ELSE 2 
        END AS semestre,
        DAYOFWEEK(job_posted_at_datetime_utc) AS dia_semana,
        DATE_FORMAT(job_posted_at_datetime_utc, 'EEEE') AS nombre_dia_semana,
        DATE_FORMAT(job_posted_at_datetime_utc, 'EEE') AS nombre_dia_semana_corto,
        WEEKOFYEAR(job_posted_at_datetime_utc) AS semana_anio,
        DAYOFYEAR(job_posted_at_datetime_utc) AS dia_anio,
        CASE 
            WHEN DAYOFWEEK(job_posted_at_datetime_utc) IN (1, 7) THEN true 
            ELSE false 
        END AS es_fin_semana,
        CASE 
            WHEN DAY(job_posted_at_datetime_utc) = 1 THEN true 
            ELSE false 
        END AS es_primer_dia_mes,
        CASE 
            WHEN DAY(job_posted_at_datetime_utc) = DAY(LAST_DAY(job_posted_at_datetime_utc)) THEN true 
            ELSE false 
        END AS es_ultimo_dia_mes,
        CURRENT_TIMESTAMP() AS fecha_carga
    FROM prueba_api.silver.jobs_silver
    WHERE job_posted_at_datetime_utc IS NOT NULL
) AS source
ON target.fecha = source.fecha

-- Si la fecha ya existe, actualizar sus datos
WHEN MATCHED THEN
    UPDATE SET
        target.anio = source.anio,
        target.mes = source.mes,
        target.dia = source.dia,
        target.nombre_mes = source.nombre_mes,
        target.nombre_mes_corto = source.nombre_mes_corto,
        target.trimestre = source.trimestre,
        target.semestre = source.semestre,
        target.dia_semana = source.dia_semana,
        target.nombre_dia_semana = source.nombre_dia_semana,
        target.nombre_dia_semana_corto = source.nombre_dia_semana_corto,
        target.semana_anio = source.semana_anio,
        target.dia_anio = source.dia_anio,
        target.es_fin_semana = source.es_fin_semana,
        target.es_primer_dia_mes = source.es_primer_dia_mes,
        target.es_ultimo_dia_mes = source.es_ultimo_dia_mes,
        target.fecha_carga = source.fecha_carga

-- Si la fecha no existe, insertarla
WHEN NOT MATCHED THEN
    INSERT (fecha, anio, mes, dia, nombre_mes, nombre_mes_corto, trimestre, semestre, dia_semana, nombre_dia_semana, nombre_dia_semana_corto, semana_anio, dia_anio, es_fin_semana, es_primer_dia_mes, es_ultimo_dia_mes, fecha_carga)
    VALUES (source.fecha, source.anio, source.mes, source.dia, source.nombre_mes, source.nombre_mes_corto, source.trimestre, source.semestre, source.dia_semana, source.nombre_dia_semana, source.nombre_dia_semana_corto, source.semana_anio, source.dia_anio, source.es_fin_semana, source.es_primer_dia_mes, source.es_ultimo_dia_mes, source.fecha_carga);

In [0]:
-- Recrear tabla de hechos principal (ahora incluye date_sk)
DROP TABLE IF EXISTS prueba_api.gold.fact_jobs;

CREATE TABLE prueba_api.gold.fact_jobs (
    -- Clave primaria
    job_id STRING,
    
    -- Claves foráneas a dimensiones
    employer_sk BIGINT,
    location_sk BIGINT,
    employment_type_sk BIGINT,
    seniority_sk BIGINT,
    date_sk BIGINT,
    
    -- Información de la oferta
    job_title STRING,
    job_description STRING,
    job_publisher STRING,
    job_onet_soc STRING,
    job_onet_job_zone STRING,
    
    -- Métricas de modalidad
    job_is_remote BOOLEAN,
    
    -- Métricas salariales
    job_salary DOUBLE,
    job_min_salary DOUBLE,
    job_max_salary DOUBLE,
    job_salary_avg DOUBLE,
    job_salary_period STRING,
    job_salary_string STRING,
    
    -- Fechas
    job_posted_at STRING,
    job_posted_at_timestamp DOUBLE,
    job_posted_at_datetime_utc TIMESTAMP,
    
    -- Aplicación
    job_apply_link STRING,
    job_apply_is_direct BOOLEAN,
    apply_options STRING,
    
    -- Beneficios
    job_benefits STRING,
    
    -- Enlaces
    job_google_link STRING,
    
    -- Metadatos
    processed_at TIMESTAMP,
    fecha_carga TIMESTAMP,
    row_hash STRING NOT NULL
)
COMMENT 'Tabla de hechos principal con todas las ofertas de trabajo y métricas';

In [0]:
-- Insertar datos en fact_jobs de forma idempotente usando MERGE
-- Se puede ejecutar múltiples veces sin duplicar registros

MERGE INTO prueba_api.gold.fact_jobs AS target
USING (
    SELECT 
        -- Clave primaria
        s.job_id,
        
        -- Claves foráneas a dimensiones (obtenidas mediante JOIN)
        e.employer_sk,
        l.location_sk,
        t.employment_type_sk,
        sen.seniority_sk,
        d.date_sk,
        
        -- Información de la oferta
        s.job_title,
        s.job_description,
        s.job_publisher,
        s.job_onet_soc,
        s.job_onet_job_zone,
        
        -- Métricas de modalidad
        s.job_is_remote,
        
        -- Métricas salariales
        s.job_salary,
        s.job_min_salary,
        s.job_max_salary,
        s.job_salary_avg,
        s.job_salary_period,
        s.job_salary_string,
        
        -- Fechas
        s.job_posted_at,
        s.job_posted_at_timestamp,
        s.job_posted_at_datetime_utc,
        
        -- Aplicación
        s.job_apply_link,
        s.job_apply_is_direct,
        s.apply_options,
        
        -- Beneficios
        s.job_benefits,
        
        -- Enlaces
        s.job_google_link,
        
        -- Metadatos
        s.processed_at,
        CURRENT_TIMESTAMP() AS fecha_carga,
        
        -- Hash de la fila (job_id + job_apply_link)
        MD5(CONCAT_WS('|', s.job_id, s.job_apply_link)) AS row_hash
        
    FROM prueba_api.silver.jobs_silver s
    LEFT JOIN prueba_api.gold.dim_employer e 
        ON s.employer_name = e.employer_name
    LEFT JOIN prueba_api.gold.dim_location l 
        ON COALESCE(s.job_country, '') = COALESCE(l.job_country, '')
        AND COALESCE(s.job_state, '') = COALESCE(l.job_state, '')
        AND COALESCE(s.job_city, '') = COALESCE(l.job_city, '')
    LEFT JOIN prueba_api.gold.dim_employment_type t 
        ON s.job_employment_type = t.job_employment_type
    LEFT JOIN prueba_api.gold.dim_seniority sen
        ON s.seniority_level = sen.seniority_level
    LEFT JOIN prueba_api.gold.dim_date d
        ON CAST(s.job_posted_at_datetime_utc AS DATE) = d.fecha
) AS source
ON target.job_id = source.job_id

-- Si el registro ya existe, actualizar los campos
WHEN MATCHED THEN
    UPDATE SET
        target.employer_sk = source.employer_sk,
        target.location_sk = source.location_sk,
        target.employment_type_sk = source.employment_type_sk,
        target.seniority_sk = source.seniority_sk,
        target.date_sk = source.date_sk,
        target.job_title = source.job_title,
        target.job_description = source.job_description,
        target.job_publisher = source.job_publisher,
        target.job_onet_soc = source.job_onet_soc,
        target.job_onet_job_zone = source.job_onet_job_zone,
        target.job_is_remote = source.job_is_remote,
        target.job_salary = source.job_salary,
        target.job_min_salary = source.job_min_salary,
        target.job_max_salary = source.job_max_salary,
        target.job_salary_avg = source.job_salary_avg,
        target.job_salary_period = source.job_salary_period,
        target.job_salary_string = source.job_salary_string,
        target.job_posted_at = source.job_posted_at,
        target.job_posted_at_timestamp = source.job_posted_at_timestamp,
        target.job_posted_at_datetime_utc = source.job_posted_at_datetime_utc,
        target.job_apply_link = source.job_apply_link,
        target.job_apply_is_direct = source.job_apply_is_direct,
        target.apply_options = source.apply_options,
        target.job_benefits = source.job_benefits,
        target.job_google_link = source.job_google_link,
        target.processed_at = source.processed_at,
        target.fecha_carga = source.fecha_carga,
        target.row_hash = source.row_hash

-- Si el registro no existe, insertarlo
WHEN NOT MATCHED THEN
    INSERT (
        job_id,
        employer_sk,
        location_sk,
        employment_type_sk,
        seniority_sk,
        date_sk,
        job_title,
        job_description,
        job_publisher,
        job_onet_soc,
        job_onet_job_zone,
        job_is_remote,
        job_salary,
        job_min_salary,
        job_max_salary,
        job_salary_avg,
        job_salary_period,
        job_salary_string,
        job_posted_at,
        job_posted_at_timestamp,
        job_posted_at_datetime_utc,
        job_apply_link,
        job_apply_is_direct,
        apply_options,
        job_benefits,
        job_google_link,
        processed_at,
        fecha_carga,
        row_hash
    )
    VALUES (
        source.job_id,
        source.employer_sk,
        source.location_sk,
        source.employment_type_sk,
        source.seniority_sk,
        source.date_sk,
        source.job_title,
        source.job_description,
        source.job_publisher,
        source.job_onet_soc,
        source.job_onet_job_zone,
        source.job_is_remote,
        source.job_salary,
        source.job_min_salary,
        source.job_max_salary,
        source.job_salary_avg,
        source.job_salary_period,
        source.job_salary_string,
        source.job_posted_at,
        source.job_posted_at_timestamp,
        source.job_posted_at_datetime_utc,
        source.job_apply_link,
        source.job_apply_is_direct,
        source.apply_options,
        source.job_benefits,
        source.job_google_link,
        source.processed_at,
        source.fecha_carga,
        source.row_hash
    );

In [0]:
-- Verificar que todo el Star Schema se creó correctamente
SELECT 
    'DIMENSIONES' AS categoria,
    'dim_employer' AS tabla,
    COUNT(*) AS registros
FROM prueba_api.gold.dim_employer

UNION ALL

SELECT 
    'DIMENSIONES' AS categoria,
    'dim_location' AS tabla,
    COUNT(*) AS registros
FROM prueba_api.gold.dim_location

UNION ALL

SELECT 
    'DIMENSIONES' AS categoria,
    'dim_employment_type' AS tabla,
    COUNT(*) AS registros
FROM prueba_api.gold.dim_employment_type

UNION ALL

SELECT 
    'DIMENSIONES' AS categoria,
    'dim_seniority' AS tabla,
    COUNT(*) AS registros
FROM prueba_api.gold.dim_seniority

UNION ALL

SELECT 
    'DIMENSIONES' AS categoria,
    'dim_date' AS tabla,
    COUNT(*) AS registros
FROM prueba_api.gold.dim_date

UNION ALL

SELECT 
    'HECHOS' AS categoria,
    'fact_jobs' AS tabla,
    COUNT(*) AS registros
FROM prueba_api.gold.fact_jobs

UNION ALL

SELECT 
    'HECHOS' AS categoria,
    'fact_jobs_summary' AS tabla,
    COUNT(*) AS registros
FROM prueba_api.gold.fact_jobs_summary

ORDER BY categoria DESC, tabla;

In [0]:
-- Esta celda documenta cómo verificar la idempotencia
-- 
-- ✅ TODAS las operaciones de inserción son ahora IDEMPOTENTES:
--
-- 🔷 Celdas 2-4: Dimensiones usan MERGE
--    - dim_employer: MERGE ON employer_name
--    - dim_location: MERGE ON (job_country, job_state, job_city)
--    - dim_employment_type: MERGE ON job_employment_type
--
-- 🔶 Celda 5: fact_jobs usa MERGE ON job_id
--
-- 🔶 Celda 6: fact_jobs_summary usa CREATE OR REPLACE TABLE
--
-- 🧪 Cómo probar la idempotencia:
-- 1. Ejecuta las celdas 2-5 una primera vez
-- 2. Ejecuta las celdas 2-5 una segunda vez
-- 3. Ejecuta la celda 7 para verificar que los conteos NO han cambiado
--
-- Resultado esperado:
-- - Los registros en cada tabla deben ser los mismos
-- - No debe haber duplicados
-- - Si ejecutas MERGE 2 veces, debe mostrar 0 inserted, X updated

SELECT 
    '✅ Todas las operaciones son idempotentes' AS mensaje,
    'Puedes ejecutar las celdas 2-5 múltiples veces sin generar duplicados' AS detalle;